In [ ]:
#Importing libraries
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import matplotlib.pyplot as plt
from scipy.stats import zscore
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.graphics.mosaicplot import mosaic
from scipy import stats

In [ ]:
#Loading the dataset
df = pd.read_csv('medical_clean_D208_II.csv')

In [ ]:
#Information about the dataset
df.info()

In [ ]:
#Selecting relevant columns for the analysis
columns = ["ReAdmis", "Overweight", "HighBlood", "Initial_days", "Doc_visits", "Initial_admin"]

In [ ]:
#Retaining only the selected columns needed for the analysis
df = df[columns]

In [ ]:
#Checking for duplicate values
print(df.duplicated())

In [ ]:
#Checking for missing values
print(df.isnull().sum())

In [ ]:
#Converting categorical variable Readmis to a numeric variable
df['ReAdmis'] = df['ReAdmis'].map({'Yes': 1, 'No': 0})

In [ ]:
#Converting categorical variable Overweight to a numeric variable
df['Overweight'] = df['Overweight'].map({'Yes': 1, 'No': 0})

In [ ]:
#Converting categorical variable HighBlood to a numeric variable
df['HighBlood'] = df['HighBlood'].map({'Yes': 1, 'No': 0})

In [ ]:
#Converting categorical variable Initial_admin to a numeric variable
df['Initial_admin'] = df['Initial_admin'].map({'Emergency Admission':2, 'Elective Admission': 1, 'Observation Admission': 0})

In [ ]:
df.info()

In [ ]:
#List of numerical columns for checking outliers
numerical_columns = ['Overweight', 'HighBlood', 'Initial_days', 'Doc_visits', 'ReAdmis']

#Identify outliers based on Z-scores
z_scores = np.abs(zscore(df[numerical_columns]))
z_score_outliers = (z_scores > 3).sum(axis=0)
print("Outliers based on Z-score method:")
print(z_score_outliers)

In [ ]:
#Checking for outliers for Doc_visits
sns.boxplot(x = 'Doc_visits', data = df)

In [ ]:
#Summary statistics for Initial days
print(df['Initial_days'].describe())

In [ ]:
# Continuous variable 1: Distribution of Initial_days
plt.hist(df['Initial_days'], bins=30, edgecolor='black')
plt.title('Distribution of Initial Hospital Days')
plt.xlabel('Initial Days')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Summary statistics for Doc_visits
print(df['Doc_visits'].describe())

In [ ]:
# Continuous variable 1: Distribution of Doc_visits
plt.hist(df['Doc_visits'], bins=30, edgecolor='black')
plt.title('Distribution of Number of Doctors Visits')
plt.xlabel('Number of doctors visits')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Summary statistics for ReAdmis
print(df['ReAdmis'].describe())

In [ ]:
#Value count for Readmission
print(df['ReAdmis'].value_counts())
#Univariate Statistics for ReAdmis variable
readmission = df['ReAdmis'].value_counts(normalize = True).mul(100).sort_values(ascending = False)
print('Relative frequency for Readmission:', readmission)

In [ ]:
#Univariate statistics for Readmission
sns.countplot(data=df, x='ReAdmis')
plt.title('Distribution of Readmission')
plt.xlabel('Readmission (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

In [ ]:
#Summary statistics for Overweight
print(df['Overweight'].describe())

In [ ]:
#Value count for Overweight
print(df['Overweight'].value_counts())
#Univariate Statistics for Overweight variable
overweight = df['Overweight'].value_counts(normalize = True).mul(100).sort_values(ascending = False)
print('Relative frequency for Overweight:', overweight)

In [ ]:
#Univariate statistics for Overweight
sns.countplot(data=df, x='Overweight')
plt.title('Distribution of Overweight')
plt.xlabel('Overweight (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

In [ ]:
#Summary statistics for HighBlood
print(df['HighBlood'].describe())

In [ ]:
#Value count for HighBlood
print(df['HighBlood'].value_counts())
#Univariate Statistics for HighBlood variable
highblood = df['HighBlood'].value_counts(normalize = True).mul(100).sort_values(ascending = False)
print('Relative frequency for High Blood Pressure:', highblood)

In [ ]:
#Univariate statistics for HighBlood
sns.countplot(data=df, x='HighBlood')
plt.title('Distribution of High Blood Pressure')
plt.xlabel('High Blood Pressure (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

In [ ]:
#Summary statistics for Initial admission
print(df['Initial_admin'].describe())

In [ ]:
#Value count for Initial Admission
print(df['Initial_admin'].value_counts())
#Univariate Statistics for Initial Admission variable
admission = df['Initial_admin'].value_counts(normalize = True).mul(100).sort_values(ascending = False)
print('Relative frequency for Initial Admission:', admission)

In [ ]:
#Univariate statistics for Initial Admission
sns.countplot(data=df, x='Initial_admin')
plt.title('Distribution of Initial Admission')
plt.xlabel('Admission (0 = Observational, 1 = Elective, 2 = Emergency)')
plt.ylabel('Count')
plt.show()

In [ ]:
#Bivariate analysis for Initial_days and Doc_visits variables
pearson, _ = stats.pearsonr(df.Initial_days, df.Doc_visits)
print("Pearson correlation coefficient:")
print(pearson)

In [ ]:
#Bivariate statistics for Initial Days vs Doctor Visits by Readmission Status
sns.scatterplot(data=df, x='Initial_days', y='Doc_visits', hue='ReAdmis', palette='coolwarm')
plt.title('Scatterplot of Initial Days vs Doctors Visits by Readmission Status')
plt.xlabel('Initial Days')
plt.ylabel('Doctors Visits')
plt.legend(title='Readmission')
plt.show()

In [ ]:
#Bivariate analysis for Readmission and Overweight
#Contingency table
contingency_table_r_o = pd.crosstab(df['ReAdmis'], df['Overweight'])

#Perform the chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(contingency_table_r_o)

#Calculate Cramér's V
#Total number of observations
n = contingency_table_r_o.sum().sum()
#Minimum dimension of the contingency table minus 1
min_dim = min(contingency_table_r_o.shape) - 1  
cramers_v = np.sqrt(chi2 / (n * min_dim))
print('Chi squared test:', chi2)
print('P-value for chi squared test:', p)
print('Cramers_v value:',cramers_v)

In [ ]:
#Bivariate Stacked Bar Chart for Readmis and Overweight
contingency_table_r_o.plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart for ReAdmis vs Overweight')
plt.xlabel('Readmission')
plt.ylabel('Count')
plt.xticks([0, 1], ['Not Readmitted', 'Readmitted'], rotation=0)
plt.legend(title='Overweight', labels=['No (0)', 'Yes (1)'])
plt.show()

In [ ]:
#Bivariate Heatmap for Readmis and Overweight
sns.heatmap(contingency_table_r_o, annot = True)
plt.title('Heatmap for ReAdmis vs Overweight')
plt.xlabel('Overweight (0 = No, 1 = Yes)')
plt.ylabel('Readmission (0 = No, 1 = Yes)')
plt.show()

In [ ]:
#Bivariate analysis for Readmission and Highblood
#Contingency table
contingency_table_r_h = pd.crosstab(df['ReAdmis'], df['HighBlood'])

#Perform the chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(contingency_table_r_h)

#Calculate Cramér's V
#Total number of observations
n = contingency_table_r_h.sum().sum()
#Minimum dimension of the contingency table minus 1
min_dim = min(contingency_table_r_h.shape) - 1  
cramers_v = np.sqrt(chi2 / (n * min_dim))
print('Chi squared test:', chi2)
print('P-value for chi squared test:', p)
print('Cramers_v value:',cramers_v)

In [ ]:
#Bivariate Stacked Bar Chart for Readmis and HighBlood
contingency_table_r_h.plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart for ReAdmis vs High Blood Pressure')
plt.xlabel('Readmission')
plt.ylabel('Count')
plt.xticks([0, 1], ['Not Readmitted', 'Readmitted'], rotation=0)
plt.legend(title='High Blood Pressure', labels=['No (0)', 'Yes (1)'])
plt.show()

In [ ]:
#Bivariate Heatmap for Readmis and HighBlood
sns.heatmap(contingency_table_r_h, annot = True)
plt.title('Heatmap for ReAdmis vs High Blood Pressure')
plt.xlabel('Highblood Pressure (0 = No, 1 = Yes)')
plt.ylabel('Readmission (0 = No, 1 = Yes)')
plt.show()

In [ ]:
#Bivariate analysis for Readmission and Initial_admin
#Contingency table
contingency_table_r_i = pd.crosstab(df['ReAdmis'], df['Initial_admin'])

#Perform the chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(contingency_table_r_i)

#Calculate Cramér's V
#Total number of observations
n = contingency_table_r_i.sum().sum()
#Minimum dimension of the contingency table minus 1
min_dim = min(contingency_table_r_i.shape) - 1  
cramers_v = np.sqrt(chi2 / (n * min_dim))
print('Chi squared test:', chi2)
print('P-value for chi squared test:', p)
print('Cramers_v value:',cramers_v)

In [ ]:
#Bivariate Stacked Bar Chart for Readmis and Initial_admin
contingency_table_r_i.plot(kind='bar', stacked=True)
plt.title('Stacked Bar Chart for ReAdmis vs Initial Admission')
plt.xlabel('Readmission')
plt.ylabel('Count')
plt.xticks([0, 1], ['Not Readmitted', 'Readmitted'], rotation=0)
plt.legend(title='Initial Admission', labels=['Observational (0)', 'Elective (1)','Emergency(2)'])
plt.show()

In [ ]:
#Bivariate Heatmap for Readmis and Initial_admin
sns.heatmap(contingency_table_r_i, annot = True)
plt.title('Heatmap for ReAdmis vs Initial Admission')
plt.xlabel('Initial Admission (0 = Observational, 1 = Elective, 2 = Emergency)')
plt.ylabel('Readmission (0 = No, 1 = Yes)')
plt.show()

In [ ]:
#Initial Logistic Regression Model
#Define dependent and independent variables
X = df[['Overweight', 'HighBlood', 'Initial_days', 'Doc_visits', 'Initial_admin']]
y = df['ReAdmis']

#Add a constant to the model
X = sm.add_constant(X)

#Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Fit the logistic regression model 
logit_model = sm.Logit(y_train, X_train)
initial_model = logit_model.fit()

#Model summary
print(initial_model.summary())

In [ ]:
#Correlation Matrix
X_corr = df[['Overweight', 'HighBlood', 'Initial_days', 'Doc_visits', 'Initial_admin']]
corr_matrix = pd.DataFrame(X_corr).corr()
print(corr_matrix)

In [ ]:
#Variance Inflation Factor (VIF)
#Define the independent variables
X_vif = df[['Overweight', 'HighBlood', 'Initial_days', 'Doc_visits', 'Initial_admin']]

#Add a constant to the independent variables matrix (for intercept in the model)
X_vif = sm.add_constant(X_vif)

#Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

#Display the VIF for each variable
print(vif_data)

In [ ]:
#Reduced Logistic Regression Model
#Define X values for the reduced model
X_reduced = df[['HighBlood', 'Initial_days','Initial_admin']]

#Add a constant to the model
X_reduced = sm.add_constant(X_reduced)

#Split the data into training and testing sets (80% train, 20% test)
X_train_reduced, X_test_reduced, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)

#Fit the logistic regression model 
logit_model_reduced = sm.Logit(y_train, X_train_reduced)
reduced_model= logit_model_reduced.fit()

#Model summary
print(reduced_model.summary())

In [ ]:
#Making predictions using the reduced model
y_pred = reduced_model.predict(X_test_reduced)
#Turning the predictions into binary predictions 0 for No and 1 for Yes
y_pred_class = (y_pred >= 0.5).astype(int)
#Print the binary prediction probability
print(y_pred_class)

In [ ]:
#Calculating Accuracy
reduced_model_accuracy = accuracy_score(y_test, y_pred_class)
print(f"Accuracy: {reduced_model_accuracy:.4f}")

In [ ]:
#Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred_class)
print(conf_matrix)

In [ ]:
#Create the ConfusionMatrixDisplay object
cm_plot = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=["Not Readmitted", "Readmitted"])
cm_plot.plot(cmap='Blues')
plt.show()

In [ ]:
coefficients = reduced_model.params

#Build the regression equation
terms = [f"{coef:.2f}*{var}" if var != "const" else f"{coef:.2f}" for var, coef in coefficients.items()]
equation = " + ".join(terms)

#Print the regression equation
print("Logistic Regression Equation:")
print(f"Readmission = {equation}")


In [ ]:
#Save cleaned dataset
new_data_path = 'output/new_medical_data_logistic_D208_II.csv'
df.to_csv(new_data_path, index=False)